# pymodal — End-to-end demo

Covers the full pipeline from synthetic structural-health data to a trained PyTorch
classifier, exercising every major API surface of the **signal** branch.

| Section | pymodal feature |
|---|---|
| 1 | Synthetic SDOF accelerance FRFs |
| 2 | `frf` — single-signal API, plotting, processing |
| 3 | `timeseries` — time-domain API, processing, collection, augmentation |
| 4 | `frf_collection` — batch operations, HDF5 persistence |
| 5 | `HDF5Dataset` + PyTorch `DataLoader` |
| 6–8 | 1-D CNN training with gradient accumulation and mixed precision |
| 10 | `IndicatorCollection` — CFDAC features, 2-D CNN, baseline comparison |

> **Environment**: `numpy`, `scipy`, `matplotlib`, `pint`, `pyFRF`, `h5py`,
> `audiomentations`, `torch` (all declared in `setup.py`).  
> `scikit-learn` is **not** required; the confusion matrix is built manually.

In [ ]:
print("pymodal", __import__("pymodal").__version__, "ready")

## 0 · Imports and device

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # suppress pint unit-stripping notices

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import pymodal
from pymodal import frf, frf_collection, timeseries, timeseries_collection

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RNG = np.random.default_rng(42)

print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")

## 1 · Synthetic data — SDOF accelerance FRFs

Three structural health states are simulated with 60 realisations each.  
Within each class, independent ±3 % scatter is applied to mass and stiffness
to mimic unit-to-unit manufacturing variability.

| Class | Label | Change |
|---|---|---|
| Healthy | 0 | — |
| Cracked | 1 | −10 % stiffness, +75 % damping |
| Loose   | 2 | +10 % mass |

Analytical accelerance:  
$$H(\omega) = \frac{-\omega^2}{k - m\omega^2 + j c\omega}$$

In [ ]:
# ── Signal parameters ─────────────────────────────────────────────────────────
FS  = 100.0             # sampling frequency [Hz]
DT  = 1.0 / FS          # time step [s]  ← pymodal stores this as 'sampling_rate'
T   = 10.0              # record duration [s]
t   = np.arange(0, T, DT)
N   = len(t)

# One-sided frequency grid (FFT of a real signal)
DF    = 1.0 / T                            # resolution [Hz]
freqs = np.arange(0, FS / 2 + DF, DF)     # 0 … 50 Hz, 501 bins
omega = 2 * np.pi * freqs

print(f"Time  : {N} samples @ {FS} Hz  →  {T} s")
print(f"Freq  : {len(freqs)} bins,  0–{freqs[-1]:.0f} Hz,  Δf = {DF} Hz")

In [ ]:
CLASS_DEFS = [
    {"label": 0, "tag": "Healthy", "m": 1.00, "k": 400.0, "zeta": 0.020},
    {"label": 1, "tag": "Cracked", "m": 1.00, "k": 360.0, "zeta": 0.035},
    {"label": 2, "tag": "Loose",   "m": 1.10, "k": 400.0, "zeta": 0.020},
]
N_PER_CLASS = 60


def sdof_accelerance(omega, m, k, zeta):
    """Analytical SDOF accelerance FRF: ẍ/F = −ω² / (k − mω² + jcω)."""
    c = 2 * zeta * np.sqrt(k * m)
    return -omega**2 / (k - m * omega**2 + 1j * c * omega)


H_all, labels_all, names_all = [], [], []

for cls in CLASS_DEFS:
    for i in range(N_PER_CLASS):
        # Independent ±3 % scatter on mass and stiffness
        mk = RNG.uniform(0.97, 1.03)
        kk = RNG.uniform(0.97, 1.03)
        H = sdof_accelerance(omega, cls["m"] * mk, cls["k"] * kk, cls["zeta"])
        H_all.append(H)
        labels_all.append(float(cls["label"]))
        names_all.append(f"{cls['tag']}_{i:03d}")

print(f"Generated {len(H_all)} FRFs  ({N_PER_CLASS} per class)")

## 2 · Single-signal API — `frf`

`frf` stores a frequency-domain measurement with full unit awareness via
[pint](https://pint.readthedocs.io/).  Measurements are always
`(n_freq, n_outputs, n_inputs)` complex arrays.

In [ ]:
# Create one representative frf object per class (used for single-signal demos)
demo_frfs = [
    frf(
        measurements=H_all[cls["label"] * N_PER_CLASS][:, np.newaxis, np.newaxis],
        freq_resolution=DF,
        measurements_units="millimeter / second**2 / newton",
        freq_units="hertz",
        method="SIMO",
        name=cls["tag"],
    )
    for cls in CLASS_DEFS
]

f0 = demo_frfs[0]
print(f"Measurements shape : {f0.measurements.shape}")
print(f"Frequency range    : {f0.freq_start} – {f0.freq_end}")
print(f"Resolution         : {f0.freq_resolution}")
print(f"DOF                : {f0.dof}")
print(f"Units              : {f0.measurements_units}")

In [ ]:
# pymodal's plot() method supports multiple formats
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
for f_obj, cls in zip(demo_frfs, CLASS_DEFS):
    freq = f_obj.freq_array.magnitude
    mag  = np.abs(f_obj.measurements[:, 0, 0].magnitude)
    ph   = np.angle(f_obj.measurements[:, 0, 0].magnitude)
    axes[0].semilogy(freq, mag,  label=cls["tag"])
    axes[1].plot    (freq, ph,   label=cls["tag"])
axes[0].set(ylabel="|H| (mm/s²/N)", title="SDOF Accelerance — three structural states")
axes[1].set(ylabel="Phase (rad)", xlabel="Frequency (Hz)")
axes[1].set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi],
                   [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"])
for ax in axes:
    ax.legend()
    ax.grid(True, linestyle=":")
plt.tight_layout()
plt.show()

In [ ]:
# ── change_freq_span : crop to a sub-band ──────────────────────────────────────
f_trimmed = demo_frfs[0].change_freq_span(new_max_freq=20.0)
print(f"Original : {len(demo_frfs[0])} lines  (0–50 Hz)")
print(f"Trimmed  : {len(f_trimmed)} lines  (0–20 Hz)")

# ── change_freq_resolution : coarsen via interpolation ────────────────────────
f_coarse = demo_frfs[0].change_freq_resolution(new_resolution=0.5)
print(f"Coarsened: {len(f_coarse)} lines  Δf = {f_coarse.freq_resolution}")

# Both operations return deep copies; the original is unchanged
assert len(demo_frfs[0]) == 501

## 3 · Time-domain API — `timeseries`

`timeseries` stores a time-domain measurement.  Below we simulate an SDOF
acceleration response to a unit impulse using modal superposition, then
demonstrate the collection and augmentation API.

> **API note**: the constructor parameter is `time_step` (Δt in seconds,
> e.g. `DT = 0.01`).  `sampling_rate` (fs in Hz = 1/Δt) is a derived
> read-only attribute, so `ts.sampling_rate == 100.0` when `time_step=0.01`.

In [ ]:
m, k, zeta = 1.0, 400.0, 0.02
omega_n = np.sqrt(k / m)
omega_d = omega_n * np.sqrt(1 - zeta**2)

# SDOF displacement impulse response h(t)
h_disp = np.exp(-zeta * omega_n * t) * np.sin(omega_d * t) / (m * omega_d)

# Unit-impulse excitation
exc_arr       = np.zeros(N)
exc_arr[0]    = 1.0

# Displacement response (convolution), then acceleration via finite difference
resp_disp = fftconvolve(exc_arr, h_disp, mode="full")[:N] * DT
resp_acc  = np.gradient(np.gradient(resp_disp, DT), DT)

print(f"Response range: [{resp_acc.min():.3f}, {resp_acc.max():.3f}] m/s²")

In [ ]:
# Create pymodal timeseries objects
# time_step = DT = 0.01 s  →  sampling_rate = 1/DT = 100 Hz (derived attribute)
ts_resp = timeseries(
    measurements=resp_acc,
    time_step=DT,
    measurements_units="meter / second**2",
    method="SIMO",
    name="SDOF_accel",
)
ts_exc = timeseries(
    measurements=exc_arr,
    time_step=DT,
    measurements_units="newton",
    method="excitation",
    name="SDOF_force",
)

print(f"Duration        : {ts_resp.time_span}")
print(f"Samples         : {len(ts_resp)}")
print(f"time_step       : {ts_resp.time_step}  (Δt in seconds)")
print(f"sampling_rate   : {ts_resp.sampling_rate}  (fs in Hz = 1/time_step)")

ax, _ = ts_resp.plot(title="SDOF Acceleration Response (impulse excitation)")
plt.show()

In [ ]:
# change_time_span — crop to first 5 s
ts_trimmed = ts_resp.change_time_span(new_max_time=5.0)
print(f"Original  : {len(ts_resp)} samples  ({ts_resp.time_span})")
print(f"Trimmed   : {len(ts_trimmed)} samples  ({ts_trimmed.time_span})")

# change_sampling_rate — halve sampling frequency (pass fs in Hz, not Δt)
ts_half = ts_resp.change_sampling_rate(new_sampling_rate=FS / 2)
print(f"Half rate : {len(ts_half)} samples  fs = {ts_half.sampling_rate} Hz")

In [ ]:
# Build a timeseries_collection from 6 noisy realisations
ts_variants = [
    timeseries(
        measurements=resp_acc + RNG.normal(0, 0.02, N),
        time_step=DT,
        measurements_units="meter / second**2",
        method="SIMO",
        name=f"run_{i:02d}",
    )
    for i in range(6)
]

ts_col = timeseries_collection(ts_variants, labels=[0.0] * 6, path="demo_ts.h5")
print(f"Before augmentation: {len(ts_col)} signals")

# AddGaussianNoise appends augmented copies in-place inside the HDF5 file
ts_col.AddGaussianNoise(min_amplitude=0.005, max_amplitude=0.02)
print(f"After  augmentation: {len(ts_col)} signals")

ts_col.close(keep=False)   # discard temporary file
print("Temporary HDF5 removed.")

## 4 · FRF collection — `frf_collection`

`frf_collection` writes every signal array directly into an HDF5 file on
construction.  All 180 FRFs (180 × 501 complex128 ≈ 1.4 MB) are stored on
disk; nothing is held in RAM except the open file handle and HDF5 dataset
references.

In [ ]:
COLL_PATH = Path("demo_frfs.h5")

# Wrap every synthesised FRF in a pymodal frf object
# Note: the collection's __init__ nullifies all attributes on frf_objects[0]
# (it becomes collection.collection_class, a zeroed-out template).
# Do not use frf_objects[0] directly after this call.
frf_objects = [
    frf(
        measurements=H[:, np.newaxis, np.newaxis],
        freq_resolution=DF,
        measurements_units="millimeter / second**2 / newton",
        freq_units="hertz",
        method="SIMO",
        name=name,
    )
    for H, name in zip(H_all, names_all)
]

collection = frf_collection(frf_objects, labels=labels_all, path=COLL_PATH)
print(f"Collection : {len(collection)} FRFs  →  {COLL_PATH}")
print(f"Labels     : {sorted({l[()] for l in collection.labels})}")

In [ ]:
# Batch-restrict every FRF to 0–25 Hz in-place (streams through HDF5)
collection.change_freq_span(new_max_freq=25.0)
n_freq_trimmed = collection.measurements[0].shape[0]
print(f"After change_freq_span: {n_freq_trimmed} frequency lines  (0–25 Hz)")

In [ ]:
# Collection plot: all 180 FRFs coloured by rainbow (magnitude)
ax, _ = collection.plot()
ax.set_title("All 180 FRFs — magnitude (0–25 Hz)")
plt.show()

## 5 · PyTorch dataset and DataLoader

`.torch_dataset()` closes the HDF5 file and wraps it in `HDF5Dataset`,
a `torch.utils.data.Dataset` that lazy-loads individual samples on demand.

**Complex → float transform**  
FRF measurements are `complex128`.  PyTorch CNNs expect `float32`, so we
supply a transform that converts complex FRFs to log-magnitude spectra:
```
(n_freq, n_out, n_in) complex128  →  (n_out×n_in, n_freq) float32
```

In [ ]:
def frf_to_tensor(x: np.ndarray) -> torch.Tensor:
    """complex128 (n_freq, n_out, n_in) → float32 (n_out*n_in, n_freq).
    
    Log-magnitude compresses the dynamic range and matches how FRF data is
    typically presented in structural health monitoring.
    """
    mag = np.abs(x).astype(np.float32)    # (n_freq, n_out, n_in)
    mag = mag.reshape(mag.shape[0], -1).T  # (n_out*n_in, n_freq)
    return torch.from_numpy(np.log1p(mag))

In [ ]:
# Close the HDF5 file and wrap it as a PyTorch Dataset
collection.torch_dataset()
dataset = collection.dataset
dataset.transform = frf_to_tensor

x0, y0 = dataset[0]
print(f"Dataset size   : {len(dataset)}")
print(f"Sample shape   : {x0.shape}   (channels, freq_lines)")
print(f"Label          : {y0}  dtype={y0.dtype}")

In [ ]:
# 80 / 20 train-val split
n_total = len(dataset)
n_train = int(0.8 * n_total)
n_val   = n_total - n_train

gen = torch.Generator().manual_seed(42)
train_ds, val_ds = torch.utils.data.random_split(dataset, [n_train, n_val],
                                                  generator=gen)

# num_workers=0: HDF5 files opened lazily per-worker can conflict on some
# filesystems even with HDF5_USE_FILE_LOCKING=FALSE.  Safe default until
# SWMR read-only mode is added (see Proposal 2).
BATCH = 4
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print(f"Batch shape : {xb.shape}  labels: {yb.tolist()}")
print(f"Train batches: {len(train_loader)}, val batches: {len(val_loader)}")

## 6 · Model — lightweight 1-D CNN

FRF magnitude spectra are treated as 1-D sequences.  Three convolutional
layers extract local spectral features; global average pooling collapses
frequency into a fixed-length embedding before the linear classifier.

In [ ]:
class FRF_CNN(nn.Module):
    """1-D CNN for FRF-based structural health classification."""

    def __init__(self, in_channels: int, n_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=7, padding=3), nn.BatchNorm1d(16), nn.ReLU(),
            nn.Conv1d(16,         32, kernel_size=5, padding=2), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32,         64, kernel_size=3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):
        return self.classifier(self.features(x).squeeze(-1))


N_CHANNELS = x0.shape[0]   # n_out * n_in  (= 1 for SIMO, 1 DOF)
N_CLASSES  = 3

model = FRF_CNN(in_channels=N_CHANNELS, n_classes=N_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters : {n_params:,}")
print(model)

## 7 · Training — gradient accumulation + mixed precision

Two techniques address the 4 GB VRAM constraint:

| Technique | Benefit | Implementation |
|---|---|---|
| **Gradient accumulation** | Effective batch size × `ACCUM` without extra VRAM | Scale loss, skip `step()` until Nth micro-batch |
| **Mixed precision (AMP)** | ~50 % VRAM reduction, faster matmuls on Ampere+ | `autocast` + `GradScaler` |

With `BATCH=4` and `ACCUM=4`, the effective batch size is **16** while only
4 samples occupy the GPU at once.

In [ ]:
EPOCHS      = 40
LR          = 1e-3
ACCUM       = 4          # gradient accumulation steps
AMP_ENABLED = DEVICE.type == "cuda"

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

train_losses, val_accs = [], []

for epoch in range(1, EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    optimizer.zero_grad()
    epoch_loss = 0.0

    for step, (x, y) in enumerate(train_loader, 1):
        x = x.to(DEVICE)
        y = y.long().to(DEVICE)

        with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
            loss = criterion(model(x), y) / ACCUM   # scale before accumulation

        scaler.scale(loss).backward()

        if step % ACCUM == 0 or step == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        epoch_loss += loss.item() * ACCUM

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    # ── Validation ────────────────────────────────────────────────────────────
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.long().to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                preds = model(x).argmax(1)
            correct += (preds == y).sum().item()

    val_acc = correct / n_val
    val_accs.append(val_acc)
    scheduler.step()

    if epoch % 8 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc:.3f}")

print("\nTraining complete.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(range(1, EPOCHS + 1), train_losses)
ax1.set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy")
ax2.plot(range(1, EPOCHS + 1), val_accs)
ax2.set(title="Validation accuracy", xlabel="Epoch", ylabel="Accuracy",
        ylim=(0, 1.05))
ax2.axhline(1.0, color="grey", linestyle=":", linewidth=0.8)
for ax in (ax1, ax2):
    ax.grid(True, linestyle=":")
plt.tight_layout()
plt.show()

## 8 · Evaluation — confusion matrix

In [ ]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
            preds = model(x).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_true.extend(y.tolist())

CLASS_NAMES = [cls["tag"] for cls in CLASS_DEFS]
n_cls = len(CLASS_NAMES)

# Manual confusion matrix (no sklearn required)
cm = np.zeros((n_cls, n_cls), dtype=int)
for t, p in zip(all_true, all_preds):
    cm[int(t), int(p)] += 1

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(n_cls)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
ax.set_yticks(range(n_cls)); ax.set_yticklabels(CLASS_NAMES)
ax.set(xlabel="Predicted", ylabel="True", title="Confusion matrix (validation)")
for i in range(n_cls):
    for j in range(n_cls):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

accuracy = np.trace(cm) / cm.sum()
print(f"Validation accuracy : {accuracy:.1%}")

## 10 · CFDAC classification — `IndicatorCollection`

CFDAC (Cross-correlation Function Damage Assessment Criterion) measures how well
the FRF response at each DOF pair in a *test* state correlates with the pristine
*reference*.  For a coupled structure the n\_dof × n\_dof matrix captures
**cross-DOF coupling shifts** that single-channel spectral analysis cannot detect:
damage to one element reshapes the global mode shapes, breaking correlations
between sensors that are unaffected in the reference state.

`frf_collection.cfdac_collection(reference)` returns an `IndicatorCollection`:
every item is an n\_dof × n\_dof complex matrix; the reference FRF is embedded
in the same HDF5 file (under `references/`) so the collection is self-contained.

We simulate the same three health states on a **coupled 8-DOF mass-spring chain**
excited at one end, then compare two classifiers on the same 180 signals:

| Model | Input | Architecture |
|---|---|---|
| Baseline | log \|H(ω)\| — (8 × n\_freq) | 1-D CNN, 8 channels |
| CFDAC    | \|CFDAC\| matrix — (1 × 8 × 8) | 2-D CNN, 1 channel |

In [ ]:
# ── 8-DOF coupled mass-spring chain — synthetic data generation ───────────
N_DOF  = 8
M_BASE = np.ones(N_DOF)              # 1 kg per DOF
K_BASE = np.full(N_DOF + 1, 2000.)  # 2000 N/m per spring

# Excitation / measurement at DOF 0.  Damage at the FAR END (spring 5,
# masses 6-7) so the DOF-0 auto-FRF sees only global mode shifts while
# the CFDAC cross-correlation matrix reveals the spatial pattern change.
# Mild severity (≤ 20 %) + high scatter (± 8 %) make the single-sensor
# problem genuinely ambiguous; the 8×8 CFDAC matrix retains sufficient
# spatial signal-to-noise for reliable classification.
CHAIN_DEFS = [
    {"label": 0, "tag": "Healthy",
     "dm": np.ones(N_DOF),
     "dk": np.ones(N_DOF + 1)},
    {"label": 1, "tag": "Crack_56",        # 20 % stiffness loss in spring 5
     "dm": np.ones(N_DOF),
     "dk": np.array([1.0, 1.0, 1.0, 1.0, 1.0, 0.80, 1.0, 1.0, 1.0])},
    {"label": 2, "tag": "Mass_67",         # 20 % mass increase at DOFs 6-7
     "dm": np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.20, 1.20]),
     "dk": np.ones(N_DOF + 1)},
]


def build_chain(m_vec, k_springs, z0=0.020):
    """Assemble M, C, K for a chain of N DOFs with N+1 spring constants."""
    N = len(m_vec)
    M_mat = np.diag(m_vec.astype(float))
    K_mat = np.zeros((N, N))
    for i in range(N):
        K_mat[i, i] += k_springs[i] + k_springs[i + 1]
        if i > 0:
            K_mat[i, i - 1] -= k_springs[i]
            K_mat[i - 1,  i] -= k_springs[i]
    omega_ref = np.sqrt(np.maximum(np.linalg.eigvalsh(K_mat), 0).mean())
    C_mat = (2 * z0 / max(omega_ref, 1e-6)) * K_mat
    return M_mat, C_mat, K_mat


def chain_frf(omega, M_mat, C_mat, K_mat, input_dof=0):
    """Accelerance FRFs (n_freq, N_DOF) from a single input DOF, vectorised."""
    N = M_mat.shape[0]
    F = np.zeros(N); F[input_dof] = 1.0
    D = (-omega**2)[:, None, None] * M_mat + 1j * omega[:, None, None] * C_mat + K_mat
    H_disp = np.linalg.solve(D, F)
    return (-omega**2)[:, None] * H_disp    # accelerance


def chain_sample(omega, dm, dk, rng):
    """One realisation: ± 8 % per-element scatter (high noise for realism)."""
    sm = rng.uniform(0.92, 1.08, N_DOF)
    sk = rng.uniform(0.92, 1.08, N_DOF + 1)
    M_mat, C_mat, K_mat = build_chain(M_BASE * dm * sm, K_BASE * dk * sk)
    return chain_frf(omega, M_mat, C_mat, K_mat)[:, :, np.newaxis]  # (n_freq, N_DOF, 1)


RNG_CHAIN = np.random.default_rng(13)
H_chain, labels_chain, names_chain = [], [], []
for cls in CHAIN_DEFS:
    for i in range(N_PER_CLASS):
        H_chain.append(chain_sample(omega, cls["dm"], cls["dk"], RNG_CHAIN))
        labels_chain.append(float(cls["label"]))
        names_chain.append(f"{cls['tag']}_{i:03d}")

print(f"Generated {len(H_chain)} coupled-chain FRFs  —  shape per signal: {H_chain[0].shape}")


In [ ]:
CHAIN_PATH = Path("demo_chain.h5")
CFDAC_PATH = Path("demo_cfdac.h5")

from pymodal import IndicatorCollection

# ── Reference FRF (first healthy sample, created before frf_collection consumes it) ──
ref_chain = frf(
    measurements=H_chain[0].copy(), freq_resolution=DF,
    measurements_units="millimeter / second**2 / newton",
    freq_units="hertz", method="SIMO", name="reference_healthy",
)

# ── Raw multi-DOF FRF collection ──────────────────────────────────────────
chain_frfs = [
    frf(measurements=H, freq_resolution=DF,
        measurements_units="millimeter / second**2 / newton",
        freq_units="hertz", method="SIMO", name=n)
    for H, n in zip(H_chain, names_chain)
]
chain_col = frf_collection(chain_frfs, labels=labels_chain, path=CHAIN_PATH)
chain_col.change_freq_span(new_max_freq=25.0)
n_freq_ch = chain_col.measurements[0].shape[0]

# ── Bake in stratified 70/15/15 train/val/test split ─────────────────────
chain_col.split(train_frac=0.70, val_frac=0.15, test_frac=0.15, seed=42)
print(f"Chain FRF collection : {len(chain_col)} signals × {N_DOF} DOFs × {n_freq_ch} lines")
print(f"  train={len(chain_col.train_indices)}  val={len(chain_col.val_indices)}  test={len(chain_col.test_indices)}")

# ── CFDAC collection: reference must be trimmed to the same span ──────────
ref_trimmed = ref_chain.change_freq_span(new_max_freq=25.0)
cfdac_col   = chain_col.cfdac_collection(ref_trimmed, path=CFDAC_PATH)

# Mirror the SAME split indices so both classifiers see identical partitions
cfdac_col.train_indices = chain_col.train_indices
cfdac_col.val_indices   = chain_col.val_indices
cfdac_col.test_indices  = chain_col.test_indices

print(f"CFDAC collection     : {len(cfdac_col)} items, shape {cfdac_col.measurements[0].shape}")
print(f"References stored    : {cfdac_col.n_references}")
print(f"Recovered reference  : '{cfdac_col.get_reference(0).name}'")


In [ ]:
# ── FRF overlay (DOF 0) + |CFDAC| heatmaps per class ─────────────────────
COLORS_CH = ["tab:blue", "tab:orange", "tab:green"]
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

freq_ch = np.linspace(0, 25, chain_col.measurements[0].shape[0])

ax_frf = axes[0]
for cls, col in zip(CHAIN_DEFS, COLORS_CH):
    idx  = int(cls["label"] * N_PER_CLASS)
    data = np.asarray(chain_col.measurements[idx][()])   # (n_freq, N_DOF, 1)
    ax_frf.semilogy(freq_ch, np.abs(data[:, 0, 0]), label=cls["tag"], color=col, lw=1.2)
ax_frf.set(title="FRF magnitude — DOF 0\n(3 class representatives)",
           xlabel="Frequency (Hz)", ylabel="|H| (mm/s²/N)")
ax_frf.legend(); ax_frf.grid(True, linestyle=":")

for ax, cls in zip(axes[1:], CHAIN_DEFS):
    idx  = int(cls["label"] * N_PER_CLASS)
    data = np.asarray(cfdac_col.measurements[idx][()])   # (N_DOF, N_DOF, 1)
    mat  = np.abs(data[:, :, 0])
    im   = ax.imshow(mat, cmap="viridis", vmin=0, vmax=1)
    ax.set(title=f"|CFDAC| — {cls['tag']}", xlabel="DOF", ylabel="DOF")
    ax.set_xticks(range(N_DOF)); ax.set_yticks(range(N_DOF))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("FRF magnitude and CFDAC matrices — test vs healthy reference", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Shared training + evaluation utilities ────────────────────────────────
def run_training(model, tr_ldr, va_ldr, n_val, epochs=40, lr=1e-3, accum=4):
    """Train model with gradient accumulation + AMP; return (losses, val_accs)."""
    crit   = nn.CrossEntropyLoss()
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    losses, accs = [], []
    for ep in range(1, epochs + 1):
        model.train(); opt.zero_grad(); ep_loss = 0.0
        for step, (x, y) in enumerate(tr_ldr, 1):
            x, y = x.to(DEVICE), y.long().to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                loss = crit(model(x), y) / accum
            scaler.scale(loss).backward()
            if step % accum == 0 or step == len(tr_ldr):
                scaler.step(opt); scaler.update(); opt.zero_grad()
            ep_loss += loss.item() * accum
        losses.append(ep_loss / len(tr_ldr))
        model.eval(); correct = 0
        with torch.no_grad():
            for x, y in va_ldr:
                x, y = x.to(DEVICE), y.long().to(DEVICE)
                with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                    correct += (model(x).argmax(1) == y).sum().item()
        accs.append(correct / n_val)
        sched.step()
        if ep % 10 == 0 or ep == 1:
            print(f"  ep {ep:3d}/{epochs}  loss={losses[-1]:.4f}  val_acc={accs[-1]:.3f}")
    return losses, accs


def eval_confusion(model, loader, n_cls=3):
    model.eval(); cm = np.zeros((n_cls, n_cls), dtype=int)
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
                preds = model(x).argmax(1).cpu()
            for t, p in zip(y.tolist(), preds.tolist()):
                cm[int(t), int(p)] += 1
    return cm

In [ ]:
# ── Baseline: 1-D CNN on single-sensor log-magnitude FRF (DOF 0 only) ─────
# Excitation and measurement are both at DOF 0.  Damage at the far end
# (springs 5-6, masses 6-7) produces only subtle changes in the DOF-0
# response, making single-sensor classification hard.
def frf_to_tensor_dof0(x: np.ndarray) -> torch.Tensor:
    """(n_freq, N_DOF, 1) complex → (1, n_freq) float32 log-magnitude at DOF 0."""
    mag = np.abs(x[:, 0, 0]).astype(np.float32)
    return torch.from_numpy(np.log1p(mag)[np.newaxis])

chain_col.torch_dataset()
chain_ds = chain_col.dataset
chain_ds.transform = frf_to_tensor_dof0

x_ch, _ = chain_ds[0]
print(f"Baseline input shape : {x_ch.shape}  (1, n_freq)  — DOF 0 only")

# Use the split baked into the collection (strictly separate sets)
tr_ch = torch.utils.data.Subset(chain_ds, chain_col.train_indices)
va_ch = torch.utils.data.Subset(chain_ds, chain_col.val_indices)
te_ch = torch.utils.data.Subset(chain_ds, chain_col.test_indices)
ldr_tr_ch = DataLoader(tr_ch, batch_size=BATCH, shuffle=True,  num_workers=0)
ldr_va_ch = DataLoader(va_ch, batch_size=BATCH, shuffle=False, num_workers=0)
ldr_te_ch = DataLoader(te_ch, batch_size=BATCH, shuffle=False, num_workers=0)

baseline_md = FRF_CNN(in_channels=1, n_classes=N_CLASSES).to(DEVICE)
n_bl = sum(p.numel() for p in baseline_md.parameters())
print(f"Baseline params      : {n_bl:,}\n--- Baseline training ---")
bl_losses, bl_accs = run_training(baseline_md, ldr_tr_ch, ldr_va_ch, len(va_ch))
cm_bl = eval_confusion(baseline_md, ldr_te_ch)
print(f"\nBaseline TEST accuracy: {np.trace(cm_bl)/cm_bl.sum():.1%}")


In [ ]:
# ── CFDAC model: 2-D CNN on |CFDAC| matrix ───────────────────────────────
class CFDAC_CNN(nn.Module):
    """2-D CNN for |CFDAC| matrix classification (input: 1 × N_DOF × N_DOF)."""
    def __init__(self, n_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(64, n_classes)

    def forward(self, x):
        return self.classifier(self.features(x).squeeze(-1).squeeze(-1))


def cfdac_to_tensor(x: np.ndarray) -> torch.Tensor:
    """(N_DOF, N_DOF, 1) complex → (1, N_DOF, N_DOF) float32 magnitude."""
    mat = np.abs(x[:, :, 0]).astype(np.float32)
    return torch.from_numpy(mat[np.newaxis])

cfdac_col.torch_dataset()
cfdac_ds = cfdac_col.dataset
cfdac_ds.transform = cfdac_to_tensor

x_cf, _ = cfdac_ds[0]
print(f"CFDAC input shape  : {x_cf.shape}  (1, N_DOF, N_DOF)")

# Same split as the FRF baseline — strictly separate sets
tr_cf = torch.utils.data.Subset(cfdac_ds, cfdac_col.train_indices)
va_cf = torch.utils.data.Subset(cfdac_ds, cfdac_col.val_indices)
te_cf = torch.utils.data.Subset(cfdac_ds, cfdac_col.test_indices)
ldr_tr_cf = DataLoader(tr_cf, batch_size=BATCH, shuffle=True,  num_workers=0)
ldr_va_cf = DataLoader(va_cf, batch_size=BATCH, shuffle=False, num_workers=0)
ldr_te_cf = DataLoader(te_cf, batch_size=BATCH, shuffle=False, num_workers=0)

cfdac_md = CFDAC_CNN(N_CLASSES).to(DEVICE)
n_cf_p   = sum(p.numel() for p in cfdac_md.parameters())
print(f"CFDAC model params : {n_cf_p:,}\n--- CFDAC CNN training ---")
cf_losses, cf_accs = run_training(cfdac_md, ldr_tr_cf, ldr_va_cf, len(va_cf))
cm_cf = eval_confusion(cfdac_md, ldr_te_cf)
print(f"\nCFDAC TEST accuracy : {np.trace(cm_cf)/cm_cf.sum():.1%}")


In [ ]:
CLASS_NAMES_CH = [cls["tag"] for cls in CHAIN_DEFS]
acc_bl = np.trace(cm_bl) / cm_bl.sum()
acc_cf = np.trace(cm_cf) / cm_cf.sum()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, cm_val, title in zip(
    axes,
    [cm_bl,                                      cm_cf],
    [f"Single-sensor FRF  ({acc_bl:.1%})",       f"CFDAC model  ({acc_cf:.1%})"],
):
    im = ax.imshow(cm_val, cmap="Blues")
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES_CH, rotation=30, ha="right")
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES_CH)
    ax.set(xlabel="Predicted", ylabel="True", title=title)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, cm_val[i, j], ha="center", va="center",
                    color="white" if cm_val[i, j] > cm_val.max() / 2 else "black")
    fig.colorbar(im, ax=ax)

fig.suptitle("TEST-set confusion matrices — Single-sensor FRF vs CFDAC\n"
             "(damage at far end: invisible to DOF-0 sensor, visible to CFDAC)",
             fontsize=11)
plt.tight_layout()
plt.show()


## 9 · Cleanup

In [ ]:
COLL_PATH.unlink(missing_ok=True)
CHAIN_PATH.unlink(missing_ok=True)
CFDAC_PATH.unlink(missing_ok=True)
print("Removed demo HDF5 files.")